## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
Github copilot

---

In [1]:
from random import randrange
import numpy as np
from sklearn.metrics import mean_squared_error, log_loss
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import normalize


def grad_check_sparse(f, x, analytic_grad, num_checks=10, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [2]:
data = load_diabetes()
X, y = data.data, data.target

In [3]:
def mse_loss_vectorized(w, b, X, y):
    """
    MSE loss function WITHOUT FOR LOOPs , NO REGULARIZATION
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    - gradient with respect to bias b
    """
    loss = 0.0
    dw = np.zeros_like(w)
    
    N = X.shape[0]
    y_pred = X.dot(w) + b
    residual = y_pred - y
    loss = np.mean(residual ** 2)
    dw = (2.0 / N) * X.T.dot(residual)
    db = (2.0 / N) * np.sum(residual)
    
    return loss, dw, np.array(db).reshape(1,)

In [4]:
def soft_threshold(x, delta):
    return np.sign(x) * np.maximum(np.abs(x) - delta, 0.0)

# Lasso Subgradient Descent

In [5]:
def l1_subgradient(w):
    """
    Subgradient of the L1 loss
    """
    dw = np.zeros_like(w)
    dw = np.sign(w)
    return dw
    

def lasso_subgradient_mse_loss_vectorized(w, b, X, y, alpha):
    """
    MSE loss function adding the subgradient for w
    """
    loss, dw, db = mse_loss_vectorized(w, b, X, y)
    # Add the subgradient to dw
    dw = dw + alpha * l1_subgradient(w)
    return loss, dw, db

In [6]:
class LassolSubgradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            idx = np.random.choice(N, batch_size, replace=True)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            self.w = self.w - learning_rate * dw
            self.b = self.b - learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
    
    
    def predict(self, X):
        return X.dot(self.w) + self.b

    def loss(self, X_batch, y_batch):
        return lasso_subgradient_mse_loss_vectorized(self.w, self.b, X_batch, y_batch, self.alpha)

In [7]:
model = LassolSubgradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 32208.678426
iteration 10000 / 200000: loss 3706.354420
iteration 20000 / 200000: loss 3377.772377
iteration 30000 / 200000: loss 2738.940435
iteration 40000 / 200000: loss 3367.117705
iteration 50000 / 200000: loss 3131.622406
iteration 60000 / 200000: loss 2432.424148
iteration 70000 / 200000: loss 2935.422962
iteration 80000 / 200000: loss 2506.717449
iteration 90000 / 200000: loss 2680.809971
iteration 100000 / 200000: loss 3250.221752
iteration 110000 / 200000: loss 2850.222493
iteration 120000 / 200000: loss 2656.851325
iteration 130000 / 200000: loss 2819.662247
iteration 140000 / 200000: loss 2916.014875
iteration 150000 / 200000: loss 2462.056532
iteration 160000 / 200000: loss 2473.476617
iteration 170000 / 200000: loss 2991.924059
iteration 180000 / 200000: loss 2525.527558
iteration 190000 / 200000: loss 3350.670404
MSE scikit-learn: 2912.5274926036195
MSE Coordinate descent model : 2888.6291486411324


In [8]:
model = LassolSubgradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 33541.250830
iteration 10000 / 200000: loss 4226.479694
iteration 20000 / 200000: loss 4075.218608
iteration 30000 / 200000: loss 4256.285452
iteration 40000 / 200000: loss 3787.596319
iteration 50000 / 200000: loss 3901.465413
iteration 60000 / 200000: loss 4039.778556
iteration 70000 / 200000: loss 3884.526070
iteration 80000 / 200000: loss 3308.664517
iteration 90000 / 200000: loss 3872.357102
iteration 100000 / 200000: loss 3725.812489
iteration 110000 / 200000: loss 3501.785454
iteration 120000 / 200000: loss 3919.357794
iteration 130000 / 200000: loss 4279.875549
iteration 140000 / 200000: loss 3995.869868
iteration 150000 / 200000: loss 3645.271587
iteration 160000 / 200000: loss 3165.690078
iteration 170000 / 200000: loss 3602.384523
iteration 180000 / 200000: loss 3289.887776
iteration 190000 / 200000: loss 4487.102611
MSE scikit-learn: 5650.290772564549
MSE Coordinate descent model : 3810.79411909569


# Lasso Proximal Gradient Descent (iterative soft thresholding algorithm or ISTA)

In [9]:
class LassoProximalGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            idx = np.random.choice(N, batch_size, replace=True)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # PROJECT THE GRADIENT FOR w USING soft_threshold
            self.w = self.w - learning_rate * dw
            self.w = soft_threshold(self.w, learning_rate * self.alpha)
            self.b = self.b - learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        return X.dot(self.w) + self.b

    def loss(self, X_batch, y_batch):
        return mse_loss_vectorized(self.w, self.b, X_batch, y_batch)

In [10]:
model = LassoProximalGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 26692.898221
iteration 10000 / 200000: loss 3528.195793
iteration 20000 / 200000: loss 3509.724557
iteration 30000 / 200000: loss 3072.350580
iteration 40000 / 200000: loss 3114.354799
iteration 50000 / 200000: loss 3075.789666
iteration 60000 / 200000: loss 2786.377016
iteration 70000 / 200000: loss 2836.198130
iteration 80000 / 200000: loss 3115.833753
iteration 90000 / 200000: loss 3107.779168
iteration 100000 / 200000: loss 2534.359552
iteration 110000 / 200000: loss 3166.545896
iteration 120000 / 200000: loss 2512.681332
iteration 130000 / 200000: loss 2873.214484
iteration 140000 / 200000: loss 3181.345514
iteration 150000 / 200000: loss 3022.506183
iteration 160000 / 200000: loss 3017.382501
iteration 170000 / 200000: loss 2914.924880
iteration 180000 / 200000: loss 3199.927406
iteration 190000 / 200000: loss 3115.670472
MSE scikit-learn: 2912.5274926036195
MSE Coordinate descent model : 2889.0536210547807


In [11]:
model = LassoProximalGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 29819.154179
iteration 10000 / 200000: loss 4904.878174
iteration 20000 / 200000: loss 4719.532431
iteration 30000 / 200000: loss 3576.418991
iteration 40000 / 200000: loss 3696.639877
iteration 50000 / 200000: loss 3731.968292
iteration 60000 / 200000: loss 3818.682512
iteration 70000 / 200000: loss 4145.163911
iteration 80000 / 200000: loss 3295.803068
iteration 90000 / 200000: loss 3692.390822
iteration 100000 / 200000: loss 3695.056283
iteration 110000 / 200000: loss 3909.521202
iteration 120000 / 200000: loss 3724.149502
iteration 130000 / 200000: loss 3970.289164
iteration 140000 / 200000: loss 3541.515594
iteration 150000 / 200000: loss 3831.093591
iteration 160000 / 200000: loss 4115.032518
iteration 170000 / 200000: loss 3615.852217
iteration 180000 / 200000: loss 4158.595820
iteration 190000 / 200000: loss 3627.999724
MSE scikit-learn: 5650.290772564549
MSE Coordinate descent model : 3811.1214570355573


# Lasso Projected Gradient Descent

In [12]:
class LassoProjectedGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w_p = None
        self.w_n = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w_p is None: # Initialization
            self.w_p = 0.001 * np.random.randn(d)
            self.w_n = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            idx = np.random.choice(N, batch_size, replace=True)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # Project for w_p and w_n
            self.w_p = self.w_p - learning_rate * (dw + self.alpha)
            self.w_n = self.w_n - learning_rate * (-dw + self.alpha)
            self.w_p = np.maximum(0.0, self.w_p)
            self.w_n = np.maximum(0.0, self.w_n)
            self.b = self.b - learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        w = self.w_p - self.w_n
        return X.dot(w) + self.b

    def loss(self, X_batch, y_batch):
        w = self.w_p - self.w_n
        loss, dw, db = mse_loss_vectorized(w, self.b, X_batch, y_batch)
        loss = loss + self.alpha * (np.sum(self.w_p) + np.sum(self.w_n))
        return loss, dw, db

In [13]:
model = LassoProjectedGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 25994.058474
iteration 10000 / 200000: loss 3257.589323
iteration 20000 / 200000: loss 3093.404197
iteration 30000 / 200000: loss 3025.202933
iteration 40000 / 200000: loss 2889.332299
iteration 50000 / 200000: loss 3237.434110
iteration 60000 / 200000: loss 3443.698400
iteration 70000 / 200000: loss 3185.131472
iteration 80000 / 200000: loss 3191.451000
iteration 90000 / 200000: loss 3408.326439
iteration 100000 / 200000: loss 3609.281968
iteration 110000 / 200000: loss 2943.056899
iteration 120000 / 200000: loss 3013.817759
iteration 130000 / 200000: loss 3337.449066
iteration 140000 / 200000: loss 2974.705955
iteration 150000 / 200000: loss 3633.522941
iteration 160000 / 200000: loss 3103.198223
iteration 170000 / 200000: loss 2889.125350
iteration 180000 / 200000: loss 3213.115898
iteration 190000 / 200000: loss 3451.265185
MSE scikit-learn: 2912.5274926036195
MSE Coordinate descent model : 2888.6421137877596


In [14]:
model = LassoProjectedGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 29084.086494
iteration 10000 / 200000: loss 5767.221678
iteration 20000 / 200000: loss 5415.155419
iteration 30000 / 200000: loss 5147.671862
iteration 40000 / 200000: loss 5352.723093
iteration 50000 / 200000: loss 4770.669472
iteration 60000 / 200000: loss 4812.107351
iteration 70000 / 200000: loss 6129.748105
iteration 80000 / 200000: loss 4939.077858
iteration 90000 / 200000: loss 5554.161454
iteration 100000 / 200000: loss 5084.821893
iteration 110000 / 200000: loss 5088.117797
iteration 120000 / 200000: loss 5427.695833
iteration 130000 / 200000: loss 4877.864935
iteration 140000 / 200000: loss 5707.748831
iteration 150000 / 200000: loss 5370.849195
iteration 160000 / 200000: loss 4946.911979
iteration 170000 / 200000: loss 4696.394636
iteration 180000 / 200000: loss 5192.689782
iteration 190000 / 200000: loss 5239.215040
MSE scikit-learn: 5650.290772564549
MSE Coordinate descent model : 3810.4700733410373


# Lasso Coordinate Descent (no intercept)

In [15]:
class LassoCoordinateDescent():
    def __init__(self, alpha=0.1):
        self.w = None
        self.alpha = alpha

    def train(self, X, y, num_iters=1000):
        N, d = X.shape
        
        if self.w is None:
            self.w = np.zeros(d)

        for _ in range(num_iters):
            for j in range(d):
                w_j = self.w[j]
                r = y - X.dot(self.w) + X[:, j] * w_j
                rho = X[:, j].dot(r)
                z = np.sum(X[:, j] ** 2)
                self.w[j] = soft_threshold(rho / z, self.alpha / z)

    def predict(self, X): 
        return X.dot(self.w)

In [16]:
model = LassoCoordinateDescent(alpha=0.1)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

MSE scikit-learn: 26057.124496145752
MSE Coordinate descent model : 26004.303657948614


In [17]:
model = LassoCoordinateDescent(alpha=2)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

MSE scikit-learn: 28794.887776106683
MSE Coordinate descent model : 26006.42248967681
